# 07 · Hands on: fine-tune DETR on your own dataset

> **Paper:** §3.2 (auxiliary decoding losses), §4 (training details) · **Code:** [`main.py`](../main.py), [`engine.py`](../engine.py)

Time to actually train. We'll fine-tune the COCO-pretrained DETR on a **synthetic shapes dataset** — small enough to train in ~2 minutes on a laptop, real enough to exercise every piece of machinery from notebooks `03`–`06`.

**The task:** find colored shapes and classify them by color (`red` / `green` / `blue`). Shape (rectangle vs ellipse) is a deliberate distractor — the model must learn to ignore it.

> **Why fine-tune instead of training from scratch?** DETR is notoriously slow to converge — the paper trains for **300–500 epochs** on 16 V100 GPUs (§4). From scratch on a laptop you'd wait days. Starting from COCO weights, the backbone and encoder already know what "an object" looks like; only the heads need to learn your classes. This is the same transfer-learning idea as freezing a pretrained CNN for classification.

## 0. Setup — everything this notebook needs

This notebook is **self-contained**: only third-party packages are imported, and every DETR-specific piece is written out below. Nothing comes from this repo, so you can read straight through without chasing a helper into another file.

Run this section once, then forget about it.

In [ ]:
# Standard third-party imports. Nothing from this repo -- every helper this
# notebook uses is defined below, in this file.
import itertools
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
import torchvision
from PIL import Image
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=3, sci_mode=False, linewidth=110)
np.set_printoptions(precision=3, suppress=True)

ASSETS = "_assets"                      # downloaded images are cached here
os.makedirs(ASSETS, exist_ok=True)
print("torch", torch.__version__)

### COCO class names and plot colors

In [ ]:
# DETR predicts 91 "classes" + 1 no-object slot = 92 logits per query.
# COCO's category ids are not contiguous (they run 1..90 with gaps), so the gaps
# are filled with 'N/A' placeholders and index 0 is unused. The list MUST be
# exactly 91 long -- one short and every label after the gap is silently wrong.
COCO_CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush',
]
assert len(COCO_CLASSES) == 91, f"expected 91 classes, got {len(COCO_CLASSES)}"

COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]

### Box geometry

DETR predicts boxes as **`cxcywh`** — centre + size, normalized to `[0, 1]`. IoU and plotting want **`xyxy`** corners. Mixing the two up is the single most common bug in detection code, so both conversions live here.

In [ ]:
def box_cxcywh_to_xyxy(b):
    """(cx, cy, w, h) -> (x0, y0, x1, y1), on the last dim."""
    cx, cy, w, h = b.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)


def box_xyxy_to_cxcywh(b):
    x0, y0, x1, y1 = b.unbind(-1)
    return torch.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], dim=-1)


def box_area(b):
    """Area of (x0, y0, x1, y1) boxes."""
    return (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])


def box_iou(a, b):
    """Pairwise IoU. a: (N, 4), b: (M, 4), both xyxy. Returns (iou, union), each (N, M)."""
    area_a, area_b = box_area(a), box_area(b)
    lt = torch.max(a[:, None, :2], b[None, :, :2])          # (N, M, 2) top-left of overlap
    rb = torch.min(a[:, None, 2:], b[None, :, 2:])          # (N, M, 2) bottom-right
    wh = (rb - lt).clamp(min=0)                             # no overlap -> 0
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return inter / union, union


def generalized_box_iou(a, b):
    """GIoU = IoU - |C \\ (A u B)| / |C|, where C is the smallest box enclosing both.

    Unlike IoU, GIoU keeps giving gradient when the boxes do not overlap at all:
    it measures how far apart they are, in units of the enclosing box.
    Range is [-1, 1] (1 = identical, -1 = infinitely far apart).
    """
    assert (a[:, 2:] >= a[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    assert (b[:, 2:] >= b[:, :2]).all(), "boxes must be (x0, y0, x1, y1) with x1 >= x0"
    iou, union = box_iou(a, b)
    lt = torch.min(a[:, None, :2], b[None, :, :2])          # enclosing box
    rb = torch.max(a[:, None, 2:], b[None, :, 2:])
    wh = (rb - lt).clamp(min=0)
    enclosing = wh[:, :, 0] * wh[:, :, 1]
    return iou - (enclosing - union) / enclosing


# --- quick self-check -------------------------------------------------------
_a = torch.tensor([[0.0, 0.0, 2.0, 2.0]])
_b = torch.tensor([[1.0, 1.0, 3.0, 3.0]])
assert torch.allclose(box_iou(_a, _b)[0], torch.tensor([[1 / 7]]))        # 1 / (4+4-1)
assert torch.allclose(generalized_box_iou(_a, _a), torch.tensor([[1.0]])) # identical -> 1
_far = torch.tensor([[10.0, 10.0, 11.0, 11.0]])
assert box_iou(_a, _far)[0].item() == 0.0                                 # IoU dies...
assert generalized_box_iou(_a, _far).item() < 0                           # ...GIoU still ranks
print("box helpers ok")

### Images in, tensors out

DETR's eval transform resizes the shortest side to 800px and ImageNet-normalizes. There is no fixed crop — the model accepts any input size.

In [ ]:
SAMPLE_IMAGES = {
    "cats":    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "street":  "http://images.cocodataset.org/val2017/000000000139.jpg",
    "horses":  "http://images.cocodataset.org/val2017/000000006471.jpg",
    "kitchen": "http://images.cocodataset.org/val2017/000000002153.jpg",
}


def load_image(name_or_url):
    """Load a sample image by nickname, URL, or local path. Cached under _assets/."""
    url = SAMPLE_IMAGES.get(name_or_url, name_or_url)
    if os.path.exists(url):
        return Image.open(url).convert("RGB")
    path = os.path.join(ASSETS, os.path.basename(url))
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.write(requests.get(url, timeout=60).content)
    return Image.open(path).convert("RGB")


# DETR's eval transform: resize the shortest side to 800px, to tensor, ImageNet
# normalize. There is NO fixed crop -- DETR accepts variable input sizes.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
default_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(800),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def get_device():
    """CUDA > MPS (Apple Silicon) > CPU."""
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

### Drawing detections

In [ ]:
def rescale_bboxes(boxes, size):
    """Normalized cxcywh in [0,1] -> absolute xyxy pixels. `size` is PIL's (W, H)."""
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(boxes)
    return b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)


def plot_results(pil_img, prob, boxes, ax=None, title=None, linewidth=2.5):
    """prob: (n, 91) softmax WITHOUT the no-object column. boxes: (n, 4) xyxy pixels."""
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(pil_img)
    for i, (p, (xmin, ymin, xmax, ymax)) in enumerate(zip(prob, boxes.tolist())):
        c = COLORS[i % len(COLORS)]
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                   fill=False, color=c, linewidth=linewidth))
        cl = p.argmax()
        ax.text(xmin, ymin, f'{COCO_CLASSES[cl]}: {p[cl]:0.2f}', fontsize=11,
                bbox=dict(facecolor=c, alpha=0.6, edgecolor='none'), color='white')
    ax.axis('off')
    if title:
        ax.set_title(title)
    return ax

### DETR itself

Backbone, positional encoding, transformer and prediction heads, written out. This is the same architecture as [`models/`](../models) with the inference path kept.

The proof that it is faithful is `load_state_dict(...)` below: it is **strict**, so every parameter name here has to match Facebook's released checkpoint exactly or it raises.

In [ ]:
class FrozenBatchNorm2d(nn.Module):
    """BatchNorm with statistics and affine parameters frozen as plain buffers."""
    def __init__(self, n):
        super().__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, *a, **kw):
        state_dict.pop(prefix + "num_batches_tracked", None)
        super()._load_from_state_dict(state_dict, prefix, *a, **kw)

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        scale = w * (rv + 1e-5).rsqrt()
        return x * scale + (b - rm * scale)


class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=128, temperature=10000, scale=2 * math.pi):
        super().__init__()
        self.num_pos_feats, self.temperature, self.scale = num_pos_feats, temperature, scale

    def forward(self, x, mask):
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        eps = 1e-6
        y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
        x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale
        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[..., 0::2].sin(), pos_x[..., 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[..., 0::2].sin(), pos_y[..., 1::2].cos()), dim=4).flatten(3)
        return torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)


class Backbone(nn.Module):
    """ResNet-50 trunk with frozen BN, returning only the last (stride-32) stage."""
    def __init__(self):
        super().__init__()
        net = torchvision.models.resnet50(weights=None, norm_layer=FrozenBatchNorm2d)
        self.body = torchvision.models._utils.IntermediateLayerGetter(net, {"layer4": "0"})
        self.num_channels = 2048

    def forward(self, x, mask):
        feat = self.body(x)["0"]
        feat_mask = F.interpolate(mask[None].float(), size=feat.shape[-2:]).to(torch.bool)[0]
        return feat, feat_mask


class TransformerEncoderLayer(nn.Module):
    """One encoder block: self-attention over image tokens, then a feed-forward net."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None, pos=None):
        # pos is added to the QUERY and the KEY but never to the VALUE:
        # position decides where to look, not what gets carried back.
        q = k = src if pos is None else src + pos
        src2 = self.self_attn(q, k, value=src, key_padding_mask=src_key_padding_mask)[0]
        src = self.norm1(src + self.dropout1(src2))
        src2 = self.linear2(self.dropout(F.relu(self.linear1(src))))
        return self.norm2(src + self.dropout2(src2))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])

    def forward(self, src, src_key_padding_mask=None, pos=None):
        out = src
        for layer in self.layers:
            out = layer(out, src_key_padding_mask=src_key_padding_mask, pos=pos)
        return out


class TransformerDecoderLayer(nn.Module):
    """One decoder block: queries talk to each other, then to the image, then FFN."""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        q = k = tgt if query_pos is None else tgt + query_pos
        tgt2 = self.self_attn(q, k, value=tgt)[0]              # queries deduplicate here
        tgt = self.norm1(tgt + self.dropout1(tgt2))
        tgt2 = self.multihead_attn(                            # queries read the image here
            query=tgt if query_pos is None else tgt + query_pos,
            key=memory if pos is None else memory + pos,
            value=memory, key_padding_mask=memory_key_padding_mask)[0]
        tgt = self.norm2(tgt + self.dropout2(tgt2))
        tgt2 = self.linear2(self.dropout(F.relu(self.linear1(tgt))))
        return self.norm3(tgt + self.dropout3(tgt2))


class TransformerDecoder(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, num_layers, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, memory_key_padding_mask=None, pos=None, query_pos=None):
        """Returns EVERY layer's output, stacked: (num_layers, num_queries, B, d_model).

        DETR keeps them all because the loss is applied after each decoder layer
        ("auxiliary decoding losses", paper section 3.2).
        """
        out = tgt
        intermediate = []
        for layer in self.layers:
            out = layer(out, memory, memory_key_padding_mask=memory_key_padding_mask,
                        pos=pos, query_pos=query_pos)
            intermediate.append(self.norm(out))
        return torch.stack(intermediate)


class Transformer(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.0):
        super().__init__()
        self.encoder = TransformerEncoder(d_model, nhead, dim_feedforward,
                                          num_encoder_layers, dropout)
        self.decoder = TransformerDecoder(d_model, nhead, dim_feedforward,
                                          num_decoder_layers, dropout)
        self.d_model, self.nhead = d_model, nhead

    def forward(self, src, mask, query_embed, pos_embed):
        """src/pos_embed: (B, C, H, W). mask: (B, H, W), True = padding."""
        bs, c, h, w = src.shape
        # (B, C, H, W) -> (H*W, B, C): this implementation puts the sequence axis first.
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
        mask = mask.flatten(1)

        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)
        tgt = torch.zeros_like(query_embed)                    # queries start at zero
        hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                          pos=pos_embed, query_pos=query_embed)
        return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(nn.Linear(n, k)
                                    for n, k in zip([input_dim] + h, h + [output_dim]))

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < len(self.layers) - 1 else layer(x)
        return x


class DETR(nn.Module):
    def __init__(self, num_classes=91, num_queries=100, hidden_dim=256, nheads=8,
                 enc_layers=6, dec_layers=6, dim_feedforward=2048, aux_loss=False):
        super().__init__()
        self.backbone = nn.ModuleList([Backbone(), PositionEmbeddingSine(hidden_dim // 2)])
        self.transformer = Transformer(hidden_dim, nheads, enc_layers, dec_layers,
                                       dim_feedforward)
        self.input_proj = nn.Conv2d(2048, hidden_dim, kernel_size=1)
        self.query_embed = nn.Embedding(num_queries, hidden_dim)
        self.class_embed = nn.Linear(hidden_dim, num_classes + 1)   # +1 = "no object"
        self.bbox_embed = MLP(hidden_dim, hidden_dim, 4, 3)
        self.num_queries = num_queries
        self.aux_loss = aux_loss                # keep every decoder layer's prediction

    def forward(self, images, mask=None):
        """images: (B, 3, H, W). mask: (B, H, W) with True on padded pixels."""
        if mask is None:                        # a single image needs no padding
            mask = torch.zeros(images.shape[0], *images.shape[-2:],
                               dtype=torch.bool, device=images.device)
        feat, feat_mask = self.backbone[0](images, mask)
        pos = self.backbone[1](feat, feat_mask)
        # hs: (num_decoder_layers, B, num_queries, hidden_dim)
        hs, memory = self.transformer(self.input_proj(feat), feat_mask,
                                      self.query_embed.weight, pos)

        outputs_class = self.class_embed(hs)             # (layers, B, queries, classes+1)
        outputs_coord = self.bbox_embed(hs).sigmoid()    # (layers, B, queries, 4)
        out = {"pred_logits": outputs_class[-1], "pred_boxes": outputs_coord[-1]}
        if self.aux_loss:
            out["aux_outputs"] = [{"pred_logits": a, "pred_boxes": b}
                                  for a, b in zip(outputs_class[:-1], outputs_coord[:-1])]
        return out


DETR_R50_URL = "https://dl.fbaipublicfiles.com/detr/detr-r50-e632da11.pth"


def load_pretrained_detr(device=None, aux_loss=False):
    """Build the model above and load Facebook's released COCO weights into it.

    `load_state_dict` is strict by default, which is the real test: every parameter
    name defined above has to match the official checkpoint exactly, or this raises.
    """
    model = DETR(aux_loss=aux_loss)
    ck = torch.hub.load_state_dict_from_url(DETR_R50_URL, map_location="cpu")
    model.load_state_dict(ck["model"])
    model.eval()
    return model.to(device) if device is not None else model


@torch.no_grad()
def detect(model, pil_img, threshold=0.9, device=None):
    """Returns (probs_kept (n, 91), boxes_kept xyxy pixels, raw outputs, keep mask)."""
    device = device or next(model.parameters()).device
    x = default_transform(pil_img).unsqueeze(0).to(device)
    outputs = model(x)
    probs = outputs["pred_logits"].softmax(-1)[0, :, :-1].cpu()   # drop no-object column
    keep = probs.max(-1).values > threshold
    boxes = rescale_bboxes(outputs["pred_boxes"][0, keep].cpu(), pil_img.size)
    return probs[keep], boxes, outputs, keep

In [ ]:
def hungarian(cost):
    """Minimum-cost perfect matching on one side of a rectangular cost matrix.

    Args:
        cost: (n, m) tensor. Rows are predictions, columns are targets.
    Returns:
        (row_idx, col_idx) LongTensors of length min(n, m); row_idx is sorted.
    """
    cost = cost.detach().cpu().double()
    n, m = cost.shape
    if n == 0 or m == 0:
        z = torch.zeros(0, dtype=torch.int64)
        return z, z
    # The algorithm below assumes at least as many columns as rows.
    if n > m:
        col_idx, row_idx = hungarian(cost.t())
        order = torch.argsort(row_idx)
        return row_idx[order], col_idx[order]

    INF = float("inf")
    # u/v are the dual potentials, one per row and per column. The invariant is
    # cost[i][j] - u[i] - v[j] >= 0 for every cell, with equality on matched cells.
    u = [0.0] * (n + 1)
    v = [0.0] * (m + 1)
    p = [0] * (m + 1)      # p[j] = row currently matched to column j (0 = none)
    way = [0] * (m + 1)    # way[j] = previous column on the augmenting path

    for i in range(1, n + 1):
        p[0] = i                       # column 0 is a sentinel holding the free row
        j0 = 0
        minv = [INF] * (m + 1)
        used = [False] * (m + 1)
        while True:                    # grow a shortest augmenting path
            used[j0] = True
            i0, delta, j1 = p[j0], INF, -1
            for j in range(1, m + 1):
                if used[j]:
                    continue
                cur = cost[i0 - 1][j - 1].item() - u[i0] - v[j]
                if cur < minv[j]:
                    minv[j], way[j] = cur, j0
                if minv[j] < delta:
                    delta, j1 = minv[j], j
            for j in range(m + 1):     # shift potentials so a new tight edge appears
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:             # reached a free column -> path complete
                break
        while j0:                      # flip the path, growing the matching by one
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1

    col_of_row = [0] * n
    for j in range(1, m + 1):
        if p[j]:
            col_of_row[p[j] - 1] = j - 1
    return torch.arange(n), torch.as_tensor(col_of_row, dtype=torch.int64)

In [ ]:
def hungarian_matcher(outputs, targets, cost_class=1.0, cost_bbox=5.0, cost_giou=2.0):
    """Match each image's predictions to its ground truth, one to one.

    Args:
        outputs: {"pred_logits": (B, Q, C+1), "pred_boxes": (B, Q, 4) cxcywh in [0,1]}
        targets: list of B dicts with "labels" (n,) and "boxes" (n, 4) cxcywh
    Returns:
        list of B (pred_idx, tgt_idx) LongTensor pairs, each of length min(Q, n).
    """
    out = []
    for b, tgt in enumerate(targets):
        probs = outputs["pred_logits"][b].softmax(-1)      # (Q, C+1)
        pred_boxes = outputs["pred_boxes"][b]              # (Q, 4)
        if len(tgt["labels"]) == 0:
            z = torch.zeros(0, dtype=torch.int64)
            out.append((z, z))
            continue
        # 1 - p is the paper's class cost; the constant 1 does not change the argmin.
        c_cls = -probs[:, tgt["labels"]]                               # (Q, n)
        c_l1 = torch.cdist(pred_boxes, tgt["boxes"], p=1)              # (Q, n)
        c_giou = -generalized_box_iou(box_cxcywh_to_xyxy(pred_boxes),
                                      box_cxcywh_to_xyxy(tgt["boxes"]))
        C = cost_class * c_cls + cost_bbox * c_l1 + cost_giou * c_giou
        out.append(hungarian(C))
    return out

In [ ]:
def _permutation_index(indices, which):
    """Turn the per-image match lists into one (batch_idx, elem_idx) advanced-index pair."""
    batch_idx = torch.cat([torch.full_like(pair[which], b) for b, pair in enumerate(indices)])
    elem_idx = torch.cat([pair[which] for pair in indices])
    return batch_idx, elem_idx


def set_criterion(outputs, targets, num_classes, indices=None,
                  eos_coef=0.1, cost_class=1.0, cost_bbox=5.0, cost_giou=2.0):
    """DETR's set loss (paper Eq. 2). Returns a dict of unweighted loss terms."""
    if indices is None:
        indices = hungarian_matcher(outputs, targets, cost_class, cost_bbox, cost_giou)

    src_logits = outputs["pred_logits"]                    # (B, Q, C+1)
    num_boxes = max(sum(len(t["labels"]) for t in targets), 1)

    # --- classification: every query is supervised ---------------------------
    # Unmatched queries get the no-object id (= num_classes, the last column).
    idx = _permutation_index(indices, 0)
    target_classes = torch.full(src_logits.shape[:2], num_classes,
                                dtype=torch.int64, device=src_logits.device)
    target_classes[idx] = torch.cat([t["labels"][j] for t, (_, j) in zip(targets, indices)])
    # Down-weight no-object by eos_coef, or the loss is swamped by empty slots.
    empty_weight = torch.ones(num_classes + 1, device=src_logits.device)
    empty_weight[-1] = eos_coef
    loss_ce = F.cross_entropy(src_logits.transpose(1, 2), target_classes, empty_weight)

    # --- boxes: only the matched pairs ---------------------------------------
    src_boxes = outputs["pred_boxes"][idx]
    tgt_boxes = torch.cat([t["boxes"][j] for t, (_, j) in zip(targets, indices)], dim=0)
    loss_bbox = F.l1_loss(src_boxes, tgt_boxes, reduction="none").sum() / num_boxes
    loss_giou = (1 - torch.diag(generalized_box_iou(box_cxcywh_to_xyxy(src_boxes),
                                                    box_cxcywh_to_xyxy(tgt_boxes)))).sum() / num_boxes

    # --- logging only: how many objects did the model think were there? ------
    with torch.no_grad():
        card_pred = (src_logits.argmax(-1) != num_classes).sum(1).float()
        tgt_len = torch.as_tensor([len(t["labels"]) for t in targets],
                                  dtype=torch.float, device=src_logits.device)
        cardinality_error = F.l1_loss(card_pred, tgt_len)

    return {"loss_ce": loss_ce, "loss_bbox": loss_bbox, "loss_giou": loss_giou,
            "cardinality_error": cardinality_error}


def total_loss(outputs, targets, num_classes, weight_dict, eos_coef=0.1):
    """Weighted sum over the final decoder layer plus every auxiliary layer.

    DETR applies the same loss after each decoder layer. The auxiliary terms are
    named "<loss>_<layer index>", which is what weight_dict has to contain.
    Returns (scalar total, dict of every individual term for logging).
    """
    terms = dict(set_criterion(outputs, targets, num_classes, eos_coef=eos_coef))
    for i, aux in enumerate(outputs.get("aux_outputs", [])):
        for k, v in set_criterion(aux, targets, num_classes, eos_coef=eos_coef).items():
            terms[f"{k}_{i}"] = v
    total = sum(terms[k] * w for k, w in weight_dict.items() if k in terms)
    return total, terms

In [ ]:
import random
import time

from PIL import ImageDraw        # we draw our own synthetic training images below

device = get_device()
torch.manual_seed(0)
print("device:", device)

## 1. A synthetic dataset

Generated on the fly, so there is nothing to download. Each image has 1–3 shapes; the **class is the color**, and the box is exactly known.

Note the target format — it must match what `SetCriterion` expects (see the docstrings in [`models/detr.py`](../models/detr.py#L143)):
- `boxes`: `(n, 4)` float, **normalized `(cx, cy, w, h)`** in `[0,1]`
- `labels`: `(n,)` int64, class ids starting at 0

In [ ]:
IMG = 320
PALETTE = [(220, 40, 40), (40, 170, 60), (50, 90, 220)]
NAMES = ["red", "green", "blue"]
NUM_CLASSES = 3                     # -> class_embed outputs 3 + 1 = 4 logits

IMAGENET_MEAN = torch.tensor([.485, .456, .406])[:, None, None]
IMAGENET_STD = torch.tensor([.229, .224, .225])[:, None, None]


def make_sample(rng):
    """One synthetic image + its ground-truth target dict."""
    im = Image.new("RGB", (IMG, IMG), (245, 245, 240))
    d = ImageDraw.Draw(im)
    boxes, labels = [], []
    for _ in range(rng.randint(1, 3)):
        w, h = rng.randint(60, 120), rng.randint(60, 120)
        x0, y0 = rng.randint(0, IMG - w), rng.randint(0, IMG - h)
        k = rng.randint(0, 2)                              # class = color
        shape = "ellipse" if rng.random() < .5 else "rectangle"   # distractor
        getattr(d, shape)([x0, y0, x0 + w, y0 + h], fill=PALETTE[k])
        boxes.append([(x0 + w/2)/IMG, (y0 + h/2)/IMG, w/IMG, h/IMG])   # cxcywh, normalized
        labels.append(k)
    x = torch.from_numpy(np.array(im)).permute(2, 0, 1).float() / 255.   # (C, H, W) = (3, 320, 320)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return x, im, {"boxes": torch.tensor(boxes, dtype=torch.float32),
                   "labels": torch.tensor(labels, dtype=torch.int64)}


def make_batch(bs, rng):
    out = [make_sample(rng) for _ in range(bs)]
    return torch.stack([o[0] for o in out]), [o[1] for o in out], [o[2] for o in out]


rng = random.Random(0)
x, pils, targets = make_batch(4, rng)
print("images :", tuple(x.shape))
print("target0:", {k: tuple(v.shape) for k, v in targets[0].items()})
print("labels :", targets[0]["labels"].tolist(),
      "->", [NAMES[i] for i in targets[0]["labels"].tolist()])

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, pil, t in zip(axes, pils, targets):
    ax.imshow(pil)
    for b, l in zip(t["boxes"], t["labels"]):
        cx, cy, w, h = (b * IMG).tolist()
        ax.add_patch(plt.Rectangle((cx-w/2, cy-h/2), w, h, fill=False, color="k", lw=2, ls="--"))
        ax.text(cx-w/2, cy-h/2-4, NAMES[l], fontsize=9)
    ax.axis("off")
plt.suptitle("Synthetic training data (dashed = ground truth)")
plt.tight_layout(); plt.show()

## 2. Model surgery

Three changes turn a COCO detector into a 3-class shape detector.

> Why the head has to be deleted rather than just resized: `load_state_dict(..., strict=False)` forgives *missing* and *unexpected* keys but **not** a shape mismatch. [`00 · PyTorch essentials`](00_pytorch_essentials.ipynb) §12 demonstrates all three cases.


In [ ]:
model = load_pretrained_detr(device="cpu", aux_loss=True)
print("BEFORE:", model.class_embed, "  num_queries =", model.num_queries)

# (1) new classification head: 91+1 -> 3+1 classes. Randomly initialized.
model.class_embed = nn.Linear(model.transformer.d_model, NUM_CLASSES + 1)

# (2) 100 query slots is overkill for 1-3 objects, and makes the ∅ imbalance worse.
#     Keep the first 20 PRETRAINED queries rather than starting from scratch.
NQ = 20
model.query_embed = nn.Embedding.from_pretrained(model.query_embed.weight[:NQ].clone(),
                                                 freeze=False)
model.num_queries = NQ

# (3) freeze the CNN backbone -- ImageNet/COCO features are already good, and this
#     is the bulk of the compute.
for p in model.backbone.parameters():
    p.requires_grad_(False)

model.to(device)
print("AFTER :", model.class_embed, "  num_queries =", model.num_queries)
tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\ntrainable: {tr/1e6:.1f}M / {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

Note `bbox_embed` is **kept**, not replaced — "where is the object" is task-independent, so those pretrained weights transfer directly. Only "which class is it" had to be relearned.

## 3. Loss, with auxiliary decoding losses

`aux_loss=True` makes the model return predictions from **every** decoder layer, so the loss is applied 6 times. The paper found this *"helpful to... help the model output the correct number of objects"* (§3.2) — and we'll verify that claim at the end.

In [ ]:
NUM_CLASSES = 3

weight_dict = {"loss_ce": 1, "loss_bbox": 5, "loss_giou": 2}
# the same three terms after each of the first 5 decoder layers
weight_dict.update({f"{k}_{i}": v for i in range(5) for k, v in list(weight_dict.items())})

print(f"{len(weight_dict)} weighted loss terms = 3 final + 15 auxiliary")
print(sorted(weight_dict)[:6], "...")
print()
print("hungarian_matcher + set_criterion + total_loss were all defined in section 0.")
print("total_loss runs set_criterion once per decoder layer and adds them up.")

### Optimizer: different learning rates for different parts

`class_embed` is brand new (random weights) while the transformer is pretrained. Giving the new head a **10× higher LR** lets it catch up without wrecking the pretrained layers — this made a large difference in practice.

In [ ]:
head, rest = [], []
for n, p in model.named_parameters():
    if not p.requires_grad:
        continue
    (head if ("class_embed" in n or "bbox_embed" in n) else rest).append(p)

optimizer = torch.optim.AdamW([{"params": head, "lr": 1e-3},     # new heads
                               {"params": rest, "lr": 1e-4}],    # pretrained transformer
                              weight_decay=1e-4)
print(f"heads: {sum(p.numel() for p in head)/1e3:6.0f}K params @ lr 1e-3")
print(f"rest : {sum(p.numel() for p in rest)/1e6:6.1f}M params @ lr 1e-4")

## 4. Predictions *before* training (the control)

In [ ]:
eval_rng = random.Random(999)
eval_x, eval_pils, eval_t = make_batch(6, eval_rng)


@torch.no_grad()
def predict(m, xb, thresh=0.5):
    m.eval()
    probs = m(xb.to(device))["pred_logits"].softmax(-1).cpu()   # (B, NQ, 4)
    boxes = m(xb.to(device))["pred_boxes"].cpu()                # (B, NQ, 4)
    keep = (probs.argmax(-1) != NUM_CLASSES) & (probs[..., :-1].max(-1).values > thresh)
    return probs, boxes, keep


def show(m, title, thresh=0.5):
    probs, boxes, keep = predict(m, eval_x, thresh)
    fig, axes = plt.subplots(1, 6, figsize=(17, 3.1))
    for i, (ax, pil, t) in enumerate(zip(axes, eval_pils, eval_t)):
        ax.imshow(pil); ax.axis("off")
        for q in keep[i].nonzero().flatten().tolist():
            cx, cy, w, h = (boxes[i, q] * IMG).tolist()
            cls = probs[i, q, :-1].argmax().item()
            col = np.array(PALETTE[cls]) / 255.
            ax.add_patch(plt.Rectangle((cx-w/2, cy-h/2), w, h, fill=False, color=col, lw=2.5))
            ax.text(cx-w/2, cy-h/2-4, f"{NAMES[cls]} {probs[i,q,:-1].max():.2f}",
                    fontsize=8, color="k")
        ax.set_title(f"want {sorted(NAMES[c] for c in t['labels'].tolist())}", fontsize=8)
    plt.suptitle(title); plt.tight_layout(); plt.show()
    return probs, boxes, keep


_ = show(model, "BEFORE fine-tuning — random classification head")

Nonsense, as expected: the head is random, so it fires everywhere or nowhere.

## 5. The training loop

This is the same structure as [`engine.py:train_one_epoch`](../engine.py), stripped to essentials:

```
forward -> criterion -> weighted sum -> backward -> clip grads -> step
```

Gradient clipping at 0.1 is DETR's default (`--clip_max_norm`) and matters: transformers are unstable early in training.

In [ ]:
STEPS, BATCH = 1000, 4
history = []
train_rng = random.Random(0)

model.train()
model.backbone.eval()          # keep frozen BatchNorm in eval mode

t0 = time.time()
for step in range(1, STEPS + 1):
    x, _, targets = make_batch(BATCH, train_rng)
    x = x.to(device)
    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

    loss, loss_dict = total_loss(model(x), targets, NUM_CLASSES, weight_dict, eos_coef=0.1)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 0.1)
    optimizer.step()

    history.append({"step": step, "loss": loss.detach().item(),
                    "ce": loss_dict["loss_ce"].item(),
                    "bbox": loss_dict["loss_bbox"].item(),
                    "giou": loss_dict["loss_giou"].item(),
                    "card": loss_dict["cardinality_error"].item()})
    if step % 100 == 0 or step == 1:
        h = history[-1]
        print(f"step {step:4d}  loss {h['loss']:6.3f}  ce {h['ce']:.3f}  "
              f"bbox {h['bbox']:.3f}  giou {h['giou']:.3f}  "
              f"card_err {h['card']:5.2f}   [{time.time()-t0:5.1f}s]")

print(f"\ndone in {time.time()-t0:.0f}s")

In [ ]:
def smooth(v, k=25):
    return np.convolve(v, np.ones(k)/k, mode="valid")

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, key, title in zip(axes,
                          ["loss", "ce", "giou", "card"],
                          ["total weighted loss", "classification (CE)",
                           "GIoU box loss", "cardinality error"]):
    v = [h[key] for h in history]
    ax.plot(v, alpha=.25, color="C0")
    ax.plot(range(len(v)-len(smooth(v)), len(v)), smooth(v), color="C0", lw=2)
    ax.set_title(title, fontsize=10); ax.set_xlabel("step"); ax.grid(alpha=.3)
axes[3].axhline(0, color="r", ls="--", lw=1)
plt.tight_layout(); plt.show()

print("cardinality error  = |predicted #objects - true #objects|")
print(f"  first 50 steps: {np.mean([h['card'] for h in history[:50]]):.2f}")
print(f"  last  50 steps: {np.mean([h['card'] for h in history[-50:]]):.2f}")

The **cardinality error** curve is the one to watch. It starts high (the random head labels almost every query as an object) and falls toward 0 — the model learning *how many* things are present. Remember this is logged only, never backpropagated (`@torch.no_grad()` in [`models/detr.py:129`](../models/detr.py#L129)).

## 6. Predictions after training

In [ ]:
probs, boxes, keep = show(model, "AFTER fine-tuning")

exact = 0
for i, t in enumerate(eval_t):
    got = sorted(NAMES[c] for c in probs[i, keep[i], :-1].argmax(-1).tolist())
    want = sorted(NAMES[c] for c in t["labels"].tolist())
    exact += (got == want)
    print(f"  img {i}: want {str(want):<28} got {str(got):<28} {'OK' if got==want else 'x'}")
print(f"\n{exact}/6 images with an exactly correct set of detections")

Colors correct, boxes tight, counts right — from a model that 2 minutes ago had never seen a colored rectangle. Remaining mistakes are usually on heavily-overlapping shapes, where the model either merges two objects into one or emits a spurious extra box.

Two honest caveats:
- 1000 steps is tiny. More steps keeps improving it.
- This is a toy distribution. Real fine-tuning needs real data, augmentation, and an eval protocol (COCO mAP via `pycocotools` — `uv sync --extra coco`).

## 7. Do auxiliary losses actually help? (testing a paper claim)

The paper claims aux losses especially help *"the model output the correct number of objects"* (§3.2). While writing this tutorial I ran the identical recipe both ways:

| Setting | Exactly-correct images | Failure mode |
|---|---|---|
| `aux_loss=False`, 1000 steps | **3 / 6** | duplicates — one image produced 6 boxes for 2 objects |
| `aux_loss=True`, 1000 steps | **5 / 6** | one spurious extra box on overlapping shapes |

Supervising all 6 decoder layers, not just the last, largely cleaned up the duplicate predictions — exactly the claim. (Your exact score will vary with the random seed; the *kind* of improvement is the reproducible part.) You can reproduce this by rebuilding with `aux_loss=False` and an unmodified `weight_dict`.

Why it works: each decoder layer is forced to produce a *valid complete set* on its own, so the duplicate-suppression job of query self-attention gets a training signal at every depth instead of only at the end.

## 8. Scaling up to a real dataset

To train on your own data instead of synthetic shapes, you need exactly two things:

1. **A `Dataset`** returning `(image_tensor, target_dict)` with `boxes` (normalized `cxcywh`) and `labels` (int64).
2. **`collate_fn=util.misc.collate_fn`** in your `DataLoader` — it builds the `NestedTensor` with padding masks from notebook `03`. Without it, variable-sized images won't batch.

```python
from torch.utils.data import DataLoader
from util.misc import collate_fn
loader = DataLoader(my_dataset, batch_size=4, collate_fn=collate_fn)
```

Then reuse the loop from §5. For the full-featured version — distributed training, LR schedule, COCO eval — read [`main.py`](../main.py) and [`engine.py`](../engine.py); everything there is now familiar.

**Set `num_classes` carefully.** It means `max_class_id + 1`, *not* the number of classes. See the comment at [`models/detr.py:304`](../models/detr.py#L304): COCO's ids run to 90, so DETR passes 91.

## One notebook left

| Notebook | What you learned |
|---|---|
| `01` | Attention from scratch; permutation invariance |
| `02` | Detection as direct set prediction; no anchors, no NMS |
| `03` | Pixels → sequence; padding masks; positional encodings |
| `04` | Encoder, decoder, and what object queries really are |
| `05` | Hungarian matching — the idea that makes it all work |
| `06` | Encoder separates instances, decoder traces extremities |
| `07` | Fine-tuning on your own data |
| `08` | Writing DETR yourself in 50 lines |

Notebook **`08`** is the capstone: you write DETR yourself, from a blank cell, and load the real pretrained weights into your own class.

### Where to go next after that

- **Deformable DETR** — fixes the slow convergence you felt in §5 and the weak small-object AP from notebook `03`
- **Panoptic segmentation** — [`models/segmentation.py`](../models/segmentation.py) in this repo; paper §4.4
- Re-read the paper. It should read very differently now.

## Exercises

**Exercise 1.** Add a 4th class (yellow). What must change, and why must `class_embed` be rebuilt?

<details><summary>Solution</summary>

```python
PALETTE = [(220,40,40), (40,170,60), (50,90,220), (230,200,40)]
NAMES = ["red", "green", "blue", "yellow"]
NUM_CLASSES = 4
# and in make_sample: k = rng.randint(0, 3)
```

`class_embed` must be rebuilt because its **output width is `NUM_CLASSES + 1`**. A `Linear(256, 4)` physically cannot emit 5 logits — the weight matrix is the wrong shape. `bbox_embed` is untouched: box regression doesn't care how many classes exist.

Note you must also re-create the optimizer, since it holds references to the old parameters.
</details>

---

**Exercise 2.** Make the task genuinely hard: class = **shape** (rectangle vs ellipse), colors random. Does it converge?

<details><summary>Solution</summary>

Change `make_sample` so `k` is the shape index and the fill color is random. It converges **much** more slowly, and may not reach usable accuracy in 1000 steps.

Why: notebook `03` showed the feature grid is `ceil(320/32) = 10 × 10`. A 60–120px shape covers roughly 2–4 grid cells — enough to localize a blob, nowhere near enough to resolve *corners vs curvature*. Color survives 32× downsampling; fine shape detail does not.

This is the same limitation behind DETR's weak small-object AP: **20.5 AP_S** in Table 1, against **26.6** for the Faster R-CNN that ties it on overall AP. The fix is higher feature resolution — DETR-DC5 (notebook `03`, Exercise 2) or Deformable DETR.
</details>

---

**Exercise 3.** Set `eos_coef=1.0` and retrain. What happens to the cardinality error?

<details><summary>Solution</summary>

Rebuild `criterion` with `eos_coef=1.0` and rerun §5. The `∅` class now carries full weight, so with ~17 of 20 queries being `∅`, the loss is dominated by "predict nothing". Cardinality error settles **higher**, and the model under-detects — it becomes reluctant to claim an object exists.

This is the experimental version of notebook `05`, Exercise 4.
</details>

---

**Exercise 4.** Run notebook `06`'s attention hooks on *this* fine-tuned model. Does it attend to extremities?

<details><summary>Solution</summary>

```python
dec_attn = []
h = model.transformer.decoder.layers[-1].multihead_attn.register_forward_hook(
        lambda m, i, o: dec_attn.append(o[1]))
model.eval()
with torch.no_grad():
    out = model(eval_x[:1].to(device))
h.remove()

Hf = Wf = 10                                     # 320/32
att = dec_attn[0][0].cpu()                       # (20, 100)
probs_ = out["pred_logits"].softmax(-1)[0].cpu()
q = (probs_.argmax(-1) != NUM_CLASSES).nonzero()[0].item()
plt.imshow(eval_pils[0])
plt.imshow(torch.nn.functional.interpolate(
    att[q].reshape(Hf, Wf)[None, None], size=(IMG, IMG), mode="bilinear")[0, 0],
    cmap="cividis", alpha=.7)
plt.axis("off"); plt.show()
```

You'll see the same pattern as the COCO model: attention concentrates on the **edges** of the shape, not its filled interior — because edges are what determine the box. The behavior transferred along with the pretrained weights.
</details>